# E-Commerce Customer Behavior and Purchase Pattern Analysis

**Dataset:** `ecommerce_customers.csv` (500 customer records, 8 columns)

## Research questions

1. What is the distribution of average session length?
2. How is time spent on the app distributed?
3. How is yearly spending distributed?
4. Is app time associated with yearly spending?
5. Is website time associated with yearly spending?
6. Is membership length associated with yearly spending?
7. Which numeric variable has the strongest correlation with yearly spending?
8. Which ten customers have the highest yearly spending?

The dataset contains customer-level engagement and yearly spending measures. It does not include individual orders, product categories, or purchase dates, so those cannot be analyzed here.

## Load and inspect the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
DATA_FILE = Path("ecommerce_customers.csv")
df = pd.read_csv(DATA_FILE)
df.head()

## Step 1 — Basic analysis
The shape reports rows and columns. `info()` shows data types and non-null counts, while `describe()` summarizes numeric columns.

In [ ]:
print("Shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
df.info()

In [ ]:
df.describe(include='all').T

### Column descriptions

- `Email`: customer email identifier.
- `Address`: customer address.
- `Avatar`: customer avatar/color label.
- `Avg. Session Length`: average session length.
- `Time on App`: time spent on the mobile app.
- `Time on Website`: time spent on the website.
- `Length of Membership`: customer membership duration.
- `Yearly Amount Spent`: yearly customer spending.

## Step 2 — Missing values
Count missing values by column. The supplied CSV has no missing values, so no filling or dropping is required.

In [ ]:
missing = df.isna().sum().to_frame("Missing values")
display(missing)
print("Total missing values:", int(df.isna().sum().sum()))
analysis_df = df.copy()

## Step 3 — Repeated values
Check exact duplicate rows and repeated customer emails. The original file has no exact duplicate rows; check the email identifier separately as well.

In [ ]:
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Repeated Email values:", int(df["Email"].duplicated().sum()))
if df.duplicated().sum() == 0:
    print("No exact duplicate rows found.")
if df["Email"].duplicated().sum() == 0:
    print("No repeated customer emails found.")

## Step 4 — Outliers
Review box plots and histograms for each numeric column. Use the IQR rule on at least two columns. Potential outliers are retained because unusually high customer engagement or spending may be genuine.

In [ ]:
numeric_cols = ["Avg. Session Length", "Time on App", "Time on Website",
                "Length of Membership", "Yearly Amount Spent"]
fig, axes = plt.subplots(len(numeric_cols), 2, figsize=(13, 4 * len(numeric_cols)))
for i, col in enumerate(numeric_cols):
    sns.boxplot(x=analysis_df[col], ax=axes[i, 0])
    axes[i, 0].set_title(f"Box plot: {col}")
    sns.histplot(analysis_df[col], kde=True, ax=axes[i, 1])
    axes[i, 1].set_title(f"Histogram: {col}")
plt.tight_layout()
plt.show()

In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["Time on App", "Yearly Amount Spent"]:
    lower, upper = iqr_bounds(analysis_df[col])
    count = ((analysis_df[col] < lower) | (analysis_df[col] > upper)).sum()
    print(f"{col}: lower={lower:.2f}, upper={upper:.2f}, outliers={count}")

**Decision:** Keep the flagged values for this descriptive analysis. IQR flags unusual values but does not prove they are data errors. Removing genuine high-spending customers could distort the business picture.

## Step 5 — Univariate analysis
Histograms show the shape of each numeric variable. The median is reported as a typical value; skewness helps describe asymmetry.

In [ ]:
summary = []
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(analysis_df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Number of customers")
    plt.tight_layout()
    plt.show()
    skew = analysis_df[col].skew()
    shape = "Right-skewed" if skew > 0.5 else ("Left-skewed" if skew < -0.5 else "Approximately symmetric")
    summary.append([col, analysis_df[col].median(), skew, shape])
display(pd.DataFrame(summary, columns=["Column", "Median (typical value)", "Skewness", "Distribution"]))

## Step 6 — Bivariate analysis
Scatter plots and a correlation heatmap show relationships. Correlation is an association measure and does not establish cause and effect.

In [ ]:
for x in ["Time on App", "Time on Website", "Length of Membership"]:
    plt.figure(figsize=(7, 4))
    sns.regplot(data=analysis_df, x=x, y="Yearly Amount Spent",
                scatter_kws={"alpha": 0.5})
    plt.title(f"{x} vs Yearly Amount Spent")
    plt.xlabel(x)
    plt.ylabel("Yearly Amount Spent")
    plt.tight_layout()
    plt.show()

In [ ]:
corr = analysis_df[numeric_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

channel_corr = pd.Series({
    "Time on App": analysis_df["Time on App"].corr(analysis_df["Yearly Amount Spent"]),
    "Time on Website": analysis_df["Time on Website"].corr(analysis_df["Yearly Amount Spent"])
})
display(channel_corr.to_frame("Correlation with yearly spending"))
print("Stronger channel relationship:", channel_corr.abs().idxmax())

## Step 7 — Answers to the research questions

In [ ]:
# Q1: Distribution of average session length
sns.histplot(analysis_df["Avg. Session Length"], kde=True)
plt.title("Q1. Average Session Length")
plt.xlabel("Average Session Length")
plt.ylabel("Number of customers")
plt.show()
print(f"Median: {analysis_df['Avg. Session Length'].median():.2f}; skewness: {analysis_df['Avg. Session Length'].skew():.2f}")

In [ ]:
# Q2: Distribution of time on app
sns.histplot(analysis_df["Time on App"], kde=True)
plt.title("Q2. Time Spent on App")
plt.xlabel("Time on App")
plt.ylabel("Number of customers")
plt.show()
print(f"Median app time: {analysis_df['Time on App'].median():.2f}")

In [ ]:
# Q3: Distribution of yearly spending
sns.histplot(analysis_df["Yearly Amount Spent"], kde=True)
plt.title("Q3. Yearly Amount Spent")
plt.xlabel("Yearly Amount Spent")
plt.ylabel("Number of customers")
plt.show()
print(f"Median yearly spending: ${analysis_df['Yearly Amount Spent'].median():.2f}")

In [ ]:
# Q4: App time and yearly spending
r = analysis_df["Time on App"].corr(analysis_df["Yearly Amount Spent"])
sns.regplot(data=analysis_df, x="Time on App", y="Yearly Amount Spent")
plt.title("Q4. App Time vs Yearly Spending")
plt.xlabel("Time on App")
plt.ylabel("Yearly Amount Spent")
plt.show()
print(f"Correlation: {r:.3f}")

In [ ]:
# Q5: Website time and yearly spending
r = analysis_df["Time on Website"].corr(analysis_df["Yearly Amount Spent"])
sns.regplot(data=analysis_df, x="Time on Website", y="Yearly Amount Spent")
plt.title("Q5. Website Time vs Yearly Spending")
plt.xlabel("Time on Website")
plt.ylabel("Yearly Amount Spent")
plt.show()
print(f"Correlation: {r:.3f}")

In [ ]:
# Q6: Membership length and yearly spending
r = analysis_df["Length of Membership"].corr(analysis_df["Yearly Amount Spent"])
sns.regplot(data=analysis_df, x="Length of Membership", y="Yearly Amount Spent")
plt.title("Q6. Membership Length vs Yearly Spending")
plt.xlabel("Length of Membership")
plt.ylabel("Yearly Amount Spent")
plt.show()
print(f"Correlation: {r:.3f}")

In [ ]:
# Q7: Strongest numeric relationship with yearly spending
spend_corr = analysis_df[numeric_cols].corr()["Yearly Amount Spent"].drop("Yearly Amount Spent")
display(spend_corr.reindex(spend_corr.abs().sort_values(ascending=False).index).to_frame("Correlation"))
strongest = spend_corr.abs().idxmax()
print(f"Strongest relationship: {strongest}, r={spend_corr[strongest]:.3f}")

In [ ]:
# Q8: Top 10 customers by yearly amount spent
top10 = analysis_df.nlargest(10, "Yearly Amount Spent")[["Email", "Yearly Amount Spent"]]
display(top10.reset_index(drop=True))
sns.barplot(data=top10, x="Email", y="Yearly Amount Spent")
plt.title("Q8. Top 10 Customers by Yearly Spending")
plt.xlabel("Customer Email")
plt.ylabel("Yearly Amount Spent")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

## Step 8 — Conclusion
The final summary below uses calculations from the supplied CSV.

In [ ]:
app_r = analysis_df["Time on App"].corr(analysis_df["Yearly Amount Spent"])
web_r = analysis_df["Time on Website"].corr(analysis_df["Yearly Amount Spent"])
spend_corr = analysis_df[numeric_cols].corr()["Yearly Amount Spent"].drop("Yearly Amount Spent")
strongest = spend_corr.abs().idxmax()
channel = "App" if abs(app_r) > abs(web_r) else "Website"
print(f"1. The dataset contains {len(analysis_df)} customer records and {len(analysis_df.columns)} columns.")
print(f"2. No missing values were found; exact duplicate rows: {analysis_df.duplicated().sum()}.")
print(f"3. Average yearly spending is ${analysis_df['Yearly Amount Spent'].mean():.2f}; median is ${analysis_df['Yearly Amount Spent'].median():.2f}.")
print(f"4. {strongest} has the strongest correlation with yearly spending (r={spend_corr[strongest]:.3f}).")
print(f"5. App-time correlation is {app_r:.3f}; website-time correlation is {web_r:.3f}, so {channel} time has the stronger relationship.")
print("6. Suggestion: use the stronger relationship as a starting point for controlled user-experience experiments; correlation alone does not prove causation.")